#Chapter 7.2 Data Transformation  
##You are working as a Machine Learning Engineer at a modern fintech startup. The company offers instant digital loans via a mobile app. Before feeding customer application data into your AI model (which predicts loan default risks), you need to build a robust Data Transformation Pipeline in Pandas.

The raw database extract (df_applications) is extremely messy. It contains duplicate sync logs, custom error place-holders, messy text inputs from unvalidated mobile forms, and text-categorical data that mathematical AI models cannot read directly.

In [10]:
import pandas as pd
import numpy as np

raw_production_dump = {
    'Application_ID': ['APP_701', 'APP_702', 'APP_703', 'APP_701', 'APP_704', 'APP_705', 'APP_702', 'APP_706', 'APP_707'],
    'Monthly_Income': [2800, -9999, 5200, 3100, 950, 7500, -9999, 1400, 4300],
    'Credit_Score': [670, 590, 740, 680, 9999, 810, 590, 620, 710],
    'Education': ['Bachelor', 'highschool', '  BACHELOR  ', 'bachelor', 'HighSchool', 'Bachelor', 'highschool', 'bachelor', 'HIGH_SCHOOL'],
    'Employment_Status': ['Full-Time', 'Part-Time', 'Full-Time', 'Full-Time', 'Freelancer', 'Full-Time', 'Part-Time', 'Freelancer', 'Full-Time']
}

df_applications = pd.DataFrame(raw_production_dump)

# Task 1: Drop Duplicates
df_applications.drop_duplicates(subset=['Application_ID'], keep='last', inplace=True)

# Task 2: Replace placeholders
df_applications.replace(-9999, np.nan, inplace=True)

# Task 3: Rename Columns
df_applications.rename(columns={'Monthly_Income': 'income', 'Credit_Score': 'credit_score', 'Education': 'education_level'}, inplace=True)

# Task 4: String Manipulation (Din 7.4 - curățare automată)
df_applications['education_level'] = df_applications['education_level'].str.strip().str.lower().str.replace('_', '')

# Task 5: Binning (Din 7.2 - pd.cut)
intervals = [0, 1200, 4000, 100000]
tiers = ['Tier_3', 'Tier_2', 'Tier_1']
df_applications['income_tier'] = pd.cut(df_applications['income'], bins=intervals, labels=tiers)

# Task 6: Filter Outliers
df_applications = df_applications[df_applications['credit_score'] <= 850]

# Task 7: One-Hot Encoding doar pe coloana țintă și concatenare
df_final = pd.get_dummies(df_applications, columns=['Employment_Status'], dtype=int)

print("=== PREPARED DATA FOR ML ===")
print(df_final)

=== PREPARED DATA FOR ML ===
  Application_ID  income  credit_score education_level income_tier  \
2        APP_703  5200.0           740        bachelor      Tier_1   
3        APP_701  3100.0           680        bachelor      Tier_2   
5        APP_705  7500.0           810        bachelor      Tier_1   
6        APP_702     NaN           590      highschool         NaN   
7        APP_706  1400.0           620        bachelor      Tier_2   
8        APP_707  4300.0           710      highschool      Tier_1   

   Employment_Status_Freelancer  Employment_Status_Full-Time  \
2                             0                            1   
3                             0                            1   
5                             0                            1   
6                             0                            0   
7                             1                            0   
8                             0                            1   

   Employment_Status_Part-Time 